# Local LangGrah RAG agent with Llama 3

让我们构建一个高级的RAG系统，所有操作都将在本地运行，为此我们将使用Ollama。

## ideas
我们将从三篇RAG论文中整合思想，构建一个RAG代理：

- 路由：自适应RAG（论文）。将问题引导至不同的检索方法
- 回退：纠正性RAG（论文）。当文档与查询不相关时，转而使用网络搜索
- 自我修正：自学习RAG（论文）。通过幻觉修正回答，或不回答问题

<img src="RAG_Agent_langGraph.png">

请注意，这将包含一些关于代理的通用概念：

- 反思：自我修正机制是一种反思形式，LangGraph 代理会对其检索和生成过程进行反思。
- 规划：控制流中所设定的流程是一种规划形式。
- 工具使用：控制流中的特定节点（例如网络搜索）将使用工具。

## 本地模型
### LLM
使用 Ollama 和 llama3：

ollama pull llama3
### 搜索
使用 Tavily

In [1]:
import os
from pprint import pprint

CUSTOM_CACHE = r'F:\Teewon\Milvue\models'
os.environ['HF_HOME'] = CUSTOM_CACHE
os.environ['HF_HUB_CACHE'] = os.path.join(CUSTOM_CACHE, 'hub')
os.environ['TRANSFORMERS_CACHE'] = os.path.join(CUSTOM_CACHE, 'transformers')
os.environ['TORCH_HOME'] = CUSTOM_CACHE

In [2]:
from dotenv import load_dotenv

load_dotenv('../../.env')

True

In [3]:
from langchain_core.globals import set_verbose,set_debug
set_verbose(True)
set_debug(True)

### LLM

In [4]:
from langchain_ollama import ChatOllama
local_llm=ChatOllama(model="llama3:latest",format="json",temperature=0)

### Index

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_milvus import Milvus
from langchain_community.embeddings import HuggingFaceEmbeddings

urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
]

docs=[WebBaseLoader(url).load() for url in urls]
docs_list=[item for sublist in docs for item in sublist]
text_splitter=RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=250,chunk_overlap=0
)
doc_splits=text_splitter.split_documents(docs_list)

# Add to Milvus
vectorstore=Milvus.from_documents(
    documents=doc_splits,
    collection_name="rag_milvus",
    embedding=HuggingFaceEmbeddings(),
    connection_args={'uri':"../../milvus_rag.db"}
)
retriever=vectorstore.as_retriever()

C:\Users\Administrator\AppData\Local\Temp\ipykernel_31900\3751291893.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.
C:\Users\Administrator\AppData\Local\Temp\ipykernel_31900\3751291893.py:23: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding=HuggingFaceEmbeddings(),
C:\Users\Administrator\AppData\Local\Temp\ipykernel_31900\3751

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

### Retrieveal Grader检索评分

In [6]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser

'''
你是一名评分员，负责判断检索到的文档与用户问题的相关性。如果文档包含与用户问题相关的关键词，则判定为相关。无需进行严格测试，目标是过滤掉错误的检索结果。

请给出“是”或“否”的二元评分，以表明文档是否与问题相关。
将二元评分以 JSON 格式输出，仅包含一个键 'score'，无需前缀或解释。

以下是检索到的文档：
{document}

以下是用户的问题：
{question}
'''
prompt=PromptTemplate(
    template="""You are a grader assessing relevance
    of a retrieved document to a user question. If the document contains keywords related to the user question,
    grade it as relevant. It does not need to be a stringent test. The goal is to filter out erroneous retrievals.

    Give a binary score 'yes' or 'no' score to indicate whether the document is relevant to the question.
    Provide the binary score as a JSON with a single key 'score' and no premable or explaination.

    Here is the retrieved document:
    {document}

    Here is the user question:
    {question}
    """,
    input_variables=["question","document"],
)

retrieval_grader=prompt|local_llm|JsonOutputParser()
question="agent memory"
docs=retriever.invoke(question)
doc_text=docs[1].page_content
print(retrieval_grader.invoke({"question":question,"document":doc_text}))

[chain/start] [chain:RunnableSequence] Entering Chain run with input:
{
  "question": "agent memory",
  "document": "Each element is an observation, an event directly provided by the agent.\n- Inter-agent communication can trigger new natural language statements.\n\n\nRetrieval model: surfaces the context to inform the agent’s behavior, according to relevance, recency and importance.\n\nRecency: recent events have higher scores\nImportance: distinguish mundane from core memories. Ask LM directly.\nRelevance: based on how related it is to the current situation / query.\n\n\nReflection mechanism: synthesizes memories into higher level inferences over time and guides the agent’s future behavior. They are higher-level summaries of past events (<- note that this is a bit different from self-reflection above)\n\nPrompt LM with 100 most recent observations and to generate 3 most salient high-level questions given a set of observations/statements. Then ask LM to answer those questions.\n\n\nPl

### Generate

In [7]:
from langchain_classic import hub
from langchain_core.output_parsers import StrOutputParser

# Prompt
'''
你是一个用于问答任务的助手。
请使用以下检索到的上下文回答问题。如果不知道答案，只需说明不知道。
最多使用三句话，保持回答简洁：
问题：{question}
上下文：{context}
回答：
'''
prompt=PromptTemplate(
    template="""You are an assistant for question-answering tasks.
    Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know.
    Use three sentences maximum and keep the answer concise:
    Question: {question}
    Context: {context}
    Answer:
    """,
    input_variables=["question","context"],
)

# Post-processing
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Chain
rag_chain=prompt|local_llm|StrOutputParser()

# Run
question="agent memory"
docs=retriever.invoke(question)
generation=rag_chain.invoke({"context":docs,"question":question})
print(generation)

[chain/start] [chain:RunnableSequence] Entering Chain run with input:
[inputs]
[chain/start] [chain:RunnableSequence > prompt:PromptTemplate] Entering Prompt run with input:
[inputs]
[chain/end] [chain:RunnableSequence > prompt:PromptTemplate] s] Exiting Prompt run with output:
[outputs]
[llm/start] [chain:RunnableSequence > llm:ChatOllama] Entering LLM run with input:
{
  "prompts": [
    "Human: You are an assistant for question-answering tasks.\n    Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know.\n    Use three sentences maximum and keep the answer concise:\n    Question: agent memory\n    Context: [Document(metadata={'pk': 34, 'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': \"LLM Powered Autonomous Agents | Lil'Log\", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT

### Hallucination Grader幻觉评分器

In [8]:
'''
你是一名评分者，需要判断一个答案是否基于一组事实。请给出一个二元评分“是”或“否”，以表明该答案是否基于或支持一组事实。将二元评分以JSON格式提供，包含单个键“score”，无前言或解释。

以下是一组事实：
{documents}

以下是答案：
{generation}
'''
prompt=PromptTemplate(
    template="""You are a grader assessing whether
    an answer is grounded in / supported by a set of facts. Give a binary score 'yes' or 'no' score to indicate
    whether the answer is grounded in / supported by a set of facts. Provide the binary score as a JSON with a
    single key 'score' and no preamble or explanation.

    Here are the facts:
    {documents}

    Here is the answer:
    {generation}
    """,
    input_variables=["generation", "documents"],
)

hallucination_grader=prompt|local_llm|JsonOutputParser()
hallucination_grader.invoke({"documents":docs,"generation":generation})

[chain/start] [chain:RunnableSequence] Entering Chain run with input:
[inputs]
[chain/start] [chain:RunnableSequence > prompt:PromptTemplate] Entering Prompt run with input:
[inputs]
[chain/end] [chain:RunnableSequence > prompt:PromptTemplate] s] Exiting Prompt run with output:
[outputs]
[llm/start] [chain:RunnableSequence > llm:ChatOllama] Entering LLM run with input:
{
  "prompts": [
    "Human: You are a grader assessing whether\n    an answer is grounded in / supported by a set of facts. Give a binary score 'yes' or 'no' score to indicate\n    whether the answer is grounded in / supported by a set of facts. Provide the binary score as a JSON with a\n    single key 'score' and no preamble or explanation.\n\n    Here are the facts:\n    [Document(metadata={'pk': 34, 'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': \"LLM Powered Autonomous Agents | Lil'Log\", 'description': 'Building agents with LLM (large language model) as its core controller is a cool conc

{'score': 'no'}

### Answer Grader

In [9]:
# Prompt
prompt=PromptTemplate(
    template="""You are a grader assessing whether an
    answer is useful to resolve a question. Give a binary score 'yes' or 'no' to indicate whether the answer is
    useful to resolve a question. Provide the binary score as a JSON with a single key 'score' and no preamble or explanation.

    Here is the answer:
    {generation}

    Here is the question: {question}
    """,
    input_variables=["generation","question"],
)
answer_grader=prompt|local_llm|JsonOutputParser()
answer_grader.invoke({"question":question,"generation":generation})

[chain/start] [chain:RunnableSequence] Entering Chain run with input:
{
  "question": "agent memory",
  "generation": "{\"agent memory\": \"The agent's memory is composed of short-term memory, which enables in-context learning, and long-term memory, which allows the agent to retain and recall information over extended periods.\"}"
}
[chain/start] [chain:RunnableSequence > prompt:PromptTemplate] Entering Prompt run with input:
{
  "question": "agent memory",
  "generation": "{\"agent memory\": \"The agent's memory is composed of short-term memory, which enables in-context learning, and long-term memory, which allows the agent to retain and recall information over extended periods.\"}"
}
[chain/end] [chain:RunnableSequence > prompt:PromptTemplate] s] Exiting Prompt run with output:
[outputs]
[llm/start] [chain:RunnableSequence > llm:ChatOllama] Entering LLM run with input:
{
  "prompts": [
    "Human: You are a grader assessing whether an\n    answer is useful to resolve a question. Give

{'score': 'yes'}

### Router

In [10]:
prompt=PromptTemplate(
    template="""You are an expert at routing a
    user question to a vectorstore or web search. Use the vectorstore for questions on LLM  agents,
    prompt engineering, and adversarial attacks. You do not need to be stringent with the keywords
    in the question related to these topics. Otherwise, use web-search. Give a binary choice 'web_search'
    or 'vectorstore' based on the question. Return the a JSON with a single key 'datasource' and
    no premable or explaination.

    Question to route:
    {question}
    """,
    input_variables=["question"],
)

question_router=prompt|local_llm|JsonOutputParser()
question="llm agent memory"
docs=retriever.invoke(question)
doc_txt=docs[1].page_content
print(question_router.invoke({"question":question}))

[chain/start] [chain:RunnableSequence] Entering Chain run with input:
{
  "question": "llm agent memory"
}
[chain/start] [chain:RunnableSequence > prompt:PromptTemplate] Entering Prompt run with input:
{
  "question": "llm agent memory"
}
[chain/end] [chain:RunnableSequence > prompt:PromptTemplate] s] Exiting Prompt run with output:
[outputs]
[llm/start] [chain:RunnableSequence > llm:ChatOllama] Entering LLM run with input:
{
  "prompts": [
    "Human: You are an expert at routing a\n    user question to a vectorstore or web search. Use the vectorstore for questions on LLM  agents,\n    prompt engineering, and adversarial attacks. You do not need to be stringent with the keywords\n    in the question related to these topics. Otherwise, use web-search. Give a binary choice 'web_search'\n    or 'vectorstore' based on the question. Return the a JSON with a single key 'datasource' and\n    no premable or explaination.\n\n    Question to route:\n    llm agent memory"
  ]
}
[llm/end] [chain:

### Search

In [11]:
from langchain_community.tools import DuckDuckGoSearchResults
web_search_tool=DuckDuckGoSearchResults()

我们将把这些作为 LangGraph 中的控制流来实现。

### State

In [12]:
from typing_extensions import TypedDict
from typing import List

class GraphState(TypedDict):
    """
    表示我们图的当前状态。

    属性：
    question：问题
    generation：LLM生成
    web_search：是否添加搜索
    documents：文档列表
    """
    question: str
    generation: str
    web_search: str
    documents: List[str]

### Nodes

In [17]:
from langchain_core.documents import Document

def retrieve(state):
    """
    从向量存储中检索文档

    参数：
    state (dict)：当前图状态

    返回：
    state (dict)：在状态中新增键 documents，包含检索到的文档
    """
    print("---RETRIEVE---")
    question=state["question"]

    # Retrieval
    documents=retriever.invoke(question)
    return {"documents":documents,"question":question}

def generate(state):
    """
    使用检索到的文档通过RAG生成答案

    参数：
    state (dict)：当前图状态

    返回：
    state (dict)：在状态中新增键 generation，包含LLM生成结果
    """
    print("---GENERATE---")
    question=state["question"]
    documents=state["documents"]

    # RAG generation
    generation=rag_chain.invoke({"context":documents,"question":question})
    return {"documents":documents,"question":question,"generation":generation}

def grade_documents(state):
    """
    判断检索到的文档是否与问题相关
    如果任何文档不相关，我们将设置一个标志以运行网络搜索

    参数：
    state (dict)：当前图状态

    返回：
    state (dict)：过滤掉不相关的文档，并更新网络搜索状态
    """
    print("---CHECK DOCUMENT RELEVANCE TO QUESTION---")
    question=state["question"]
    documents=state["documents"]

    # Score each doc
    filtered_docs=[]
    web_search="No"
    for d in documents:
        score=retrieval_grader.invoke({"question":question,"document":d.page_content})
        grade=score['score']
        # Document relevant
        if grade.lower()=='yes':
            print("---GRADE: DOCUMENT RELEVANT---")
            filtered_docs.append(d)
        # Document not relevant
        else:
            print("---GRADE: DOCUMENT NOT RELEVANT---")
            # We do no include the document in filtered_docs
            # We set a flag to indicate that we want to run web search
            web_search="Yes"
            continue
    return {"documents":filtered_docs,"question":question,"web_search":web_search}

def web_search(state):
    """
    基于问题的网页搜索

    参数：
    state (dict)：当前图状态

    返回：
    state (dict)：将网页结果附加到文档中
    """
    print("---WEB-SEARCH---")
    question=state["question"]
    documents=state.get("documents",[])

    # Web Search
    web_content=web_search_tool.invoke({"query":question})
    # DuckDuckGoSearchResults 返回的是字符串
    # web_results="\n".join([d["content"] for d in docs])
    #web_results=Document(page_content=web_results)
    # if documents is not None:
    #     documents.append(web_results)
    # else:
    #     documents=[web_results]
    # return  {"documents":documents,"question":question}
    web_doc=Document(page_content=web_content)
    documents.append(web_doc)
    return {"documents":documents,"question":question}

In [30]:
result = web_search_tool.invoke({"query": "test"})
print(type(result))
print(result)

Error in ConsoleCallbackHandler.on_tool_start callback: KeyError('input')


[tool/end] [tool:duckduckgo_results_json] [1.06s] Exiting Tool run with output:
"snippet: June 16, 2026 - Look up test, testing, Test, or TEST in Wiktionary, the free dictionary. ... Test (assessment), an educational assessment intended to measure the respondents' knowledge or other abilities ... Test., abbreviation for Testament, referring to the Old Testament and New Testament of the Christian Bible., title: Test - Wikipedia, link: https://en.wikipedia.org/wiki/Test, snippet: A test is a procedure, method, or examination designed to evaluate, assess, or measure the qualities, performance, knowledge, abilities, reliability, or other attributes of a person, object, system, or concept., title: Test, link: https://grokipedia.com/page/Test, snippet: 1 week ago - a means of testing: such as; something (such as a series of questions or exercises) for measuring the skill, knowledge, intelligence, capacities, or aptitudes of an individual or group… See the full definition, title: TEST Definit

### Conditional edge

In [18]:
def route_question(state):
    """
    将问题路由到网络搜索或RAG。

    参数：
    state (dict)：当前图状态

    返回：
    str：要调用的下一个节点
    """
    print("---ROUTE QUESTION---")
    question=state["question"]
    print(question)
    source=question_router.invoke({"question":question})
    print(source)
    print(source["datasource"])
    if source['datasource']=="web_search":
        print("---ROUTE QUESTION TO WEB-SEARCH---")
        return "websearch"
    elif source['datasource']=="vectorstore":
        print("---ROUTE QUESTION TO VECTORSTORE---")
        return "vectorstore"

def decide_to_generate(state):
    """
    确定是否生成回答，或添加网络搜索

    参数：
    state (dict)：当前图状态

    返回：
    str：下个节点调用的二元决策
    """
    print("---ASSESS GRADED DOCUMENTS---")
    question=state["question"]
    web_search=state["web_search"]
    filtered_documents=state["documents"]

    if web_search=="Yes":
        # All documents have been filtered chech_relevant
        # We will re-generate a new query
        print("---DECISION: ALL DOCUMENTS ARE NOT RELEVANT TO QUESTION, INCLUDE WEB SEARCH---")
        return "websearch"
    else:
        # We have relevant documents, so generate answer
        print("---DECISION: GENERATE")
        return "generate"

def grade_generation_v_documents_and_question(state):
    """
    判断该生成是否基于文档，并回答问题。

    参数：
    state (dict)：当前图状态

    返回：
    str：下一步调用节点的决策
    """
    print("---CHECH HALLUCINATIONS---")
    question=state["question"]
    documents=state["documents"]
    generation=state["generation"]

    score=hallucination_grader.invoke({"documents":documents,"generation":generation})
    grade=score["score"]

    # Check hallucination
    if grade=="yes":
        print("---DECISION: GENERATION IS GROUNDED IN DOCUMENT---")
        # Check question-answering
        print("---GRADE GENERATION vs QUESTION---")
        score=answer_grader.invoke({"question":question,"generation":generation})
        grade=score['score']
        if grade=="yes":
            print("---DECISION: GENERATION ADDRESSES QUESTION---")
            return "useful"
        else:
            print("---DECISION: GENERATION DOSE NOT ADDRESS QUESTION---")
            return "not useful"
    else:
        pprint("---DECISION: GENERATION IS NOT GROUNDED IN DOCUMENT, RE-TRY---")
        return "not supported"

from langgraph.graph import END,StateGraph
workflow=StateGraph(GraphState)

# Define the nodes
workflow.add_node("websearch",web_search)
workflow.add_node("retrieve",retrieve)
workflow.add_node("grade_document",grade_documents)
workflow.add_node("generate",generate)

### Graph Build

In [19]:
workflow.set_conditional_entry_point(
    route_question,
    {
        "websearch":"websearch",
        "vectorstore":"retrieve",
    },
)

workflow.add_edge("retrieve","grade_document")
workflow.add_conditional_edges(
    "grade_document",
    decide_to_generate,
    {
        "websearch":"websearch",
        "generate":"generate",
    },
)
workflow.add_edge("websearch","generate")
workflow.add_conditional_edges(
    "generate",
    grade_generation_v_documents_and_question,
    {
        "not supported":"generate",
        "useful":END,
        "not useful":"websearch",
    }
)

In [26]:
# Compile
app=workflow.compile()

# Test
from pprint import pprint
# 代理内存的类型有哪些？
inputs={"question":"What are the types of agent memory?"}
for output in app.stream(inputs):
    for key,value in output.items():
        pprint(f"Finished running: {key}")
pprint(value["generation"])

[chain/start] [chain:LangGraph] Entering Chain run with input:
{
  "question": "What are the types of agent memory?"
}
[chain/start] [chain:LangGraph > chain:__start__] Entering Chain run with input:
{
  "question": "What are the types of agent memory?"
}
[chain/start] [chain:LangGraph > chain:__start__ > chain:route_question] Entering Chain run with input:
{
  "question": "What are the types of agent memory?"
}
---ROUTE QUESTION---
What are the types of agent memory?
[chain/start] [chain:LangGraph > chain:__start__ > chain:route_question > chain:RunnableSequence] Entering Chain run with input:
{
  "question": "What are the types of agent memory?"
}
[chain/start] [chain:LangGraph > chain:__start__ > chain:route_question > chain:RunnableSequence > prompt:PromptTemplate] Entering Prompt run with input:
{
  "question": "What are the types of agent memory?"
}
[chain/end] [chain:LangGraph > chain:__start__ > chain:route_question > chain:RunnableSequence > prompt:PromptTemplate] s] Exiting P

**输出完全符合预期**！🎉


### ✅ 工作流执行概览

| 节点 | 执行情况 | 结果 |
|------|----------|------|
| **路由** (`route_question`) | 问题被正确识别为关于 **LLM agents**，路由到 `vectorstore` | `{"datasource": "vectorstore"}` |
| **检索** (`retrieve`) | 从 Milvus 检索到 5 个相关文档块 | 成功返回文档列表 |
| **文档评分** (`grade_document`) | 所有 5 个文档均被判定为 **相关** (`score: yes`) | 未触发 `web_search` 回退 |
| **生成** (`generate`) | LLM 基于上下文生成答案，输出 JSON 格式 | `{ "types of agent memory": ["Short-term memory", "Long-term memory"] }` |
| **幻觉检查** (`hallucination_grader`) | 答案被判定为 **基于事实** (`score: yes`) | 通过检查 |
| **答案评估** (`answer_grader`) | 答案被判定为 **有用**（直接回答了问题）(`score: yes`) | 通过检查 |
| **最终决策** | `grade_generation_v_documents_and_question` 返回 `"useful"`，进入 `END` | 流程结束，输出最终答案 |

### 🔍 额外观察

- **路由判断准确**：问题包含 “agent memory”，被判定为 LLM agents 相关，走向量检索而非网络搜索，这是合理的。
- **文档评分无遗漏**：所有检索到的文档都与 “agent memory” 相关，因此没有触发网络搜索回退。
- **幻觉检查通过**：答案确实可以从检索到的文档（如“Short-term memory”、“Long-term memory”的描述）中找到依据。
- **答案评估通过**：直接回答了“types of agent memory”的问题。


### 44min

In [ ]:
# Compile
app = workflow.compile()

# Test
from pprint import pprint
# 在NFL选秀中，谁预计会成为首轮人选？
inputs = {"question": "Who are the Bears expected to draft first in the NFL draft?"}
for output in app.stream(inputs):
    for key, value in output.items():
        pprint(f"Finished running: {key}:")
pprint(value["generation"])